In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from pyspark.sql import SparkSession
import re
import numpy as np
import plotly.graph_objects as go
from pyspark.sql.functions import col, split, explode, regexp_replace, transform, when
from pyspark.sql import functions as F
from pyspark.sql.functions import col, monotonically_increasing_id

np.random.seed(42)

pio.renderers.default = "notebook"

# Initialize Spark Session
spark = SparkSession.builder.appName("LightcastData").getOrCreate()

# Load Data
df = spark.read.option("header", "true").option("inferSchema", "true").option("multiLine","true").option("escape", "\"").csv("../data/lightcast_job_postings.csv")
df.createOrReplaceTempView("job_postings")

# Show Schema and Sample Data
print("---This is Diagnostic check, No need to print it in the final doc---")

df.printSchema() # comment this line when rendering the submission
df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 22:37:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 22:37:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


---This is Diagnostic check, No need to print it in the final doc---
root
 |-- ID: string (nullable = true)
 |-- LAST_UPDATED_DATE: string (nullable = true)
 |-- LAST_UPDATED_TIMESTAMP: timestamp (nullable = true)
 |-- DUPLICATES: integer (nullable = true)
 |-- POSTED: string (nullable = true)
 |-- EXPIRED: string (nullable = true)
 |-- DURATION: integer (nullable = true)
 |-- SOURCE_TYPES: string (nullable = true)
 |-- SOURCES: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- ACTIVE_URLS: string (nullable = true)
 |-- ACTIVE_SOURCES_INFO: string (nullable = true)
 |-- TITLE_RAW: string (nullable = true)
 |-- BODY: string (nullable = true)
 |-- MODELED_EXPIRED: string (nullable = true)
 |-- MODELED_DURATION: integer (nullable = true)
 |-- COMPANY: integer (nullable = true)
 |-- COMPANY_NAME: string (nullable = true)
 |-- COMPANY_RAW: string (nullable = true)
 |-- COMPANY_IS_STAFFING: boolean (nullable = true)
 |-- EDUCATION_LEVELS: string (nullable = true)
 |-- EDUCATIO

+--------------------+-----------------+----------------------+----------+--------+---------+--------+--------------------+--------------------+--------------------+-----------+-------------------+--------------------+--------------------+---------------+----------------+--------+--------------------+-----------+-------------------+----------------+---------------------+-------------+-------------------+-------------+------------------+---------------+--------------------+--------------------+--------------------+-------------+------+-----------+----------------+-------------------+---------+-----------+--------------------+--------------------+-------------+------+--------------+-----+--------------------+-----+----------+---------------+--------------------+---------------+--------------------+------------+--------------------+------------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+--------------------+------+------

## Data Cleaning

### 1.1 Casting salary and experience columns

In [2]:
from pyspark.sql.functions import col

df = df.withColumn("SALARY", col("SALARY").cast("double")) \
       .withColumn("SALARY_FROM", col("SALARY_FROM").cast("double")) \
       .withColumn("SALARY_TO", col("SALARY_TO").cast("double")) \
       .withColumn("MAX_YEARS_EXPERIENCE", col("MAX_YEARS_EXPERIENCE").cast("double")) \
       .withColumn("MIN_YEARS_EXPERIENCE", col("MIN_YEARS_EXPERIENCE").cast("double"))

### 1.2 Computing medians

In [3]:
from pyspark.sql.functions import expr

median_from = df.filter((col("SALARY_FROM").isNotNull()) & (col("SALARY_FROM") > 0)) \
                .selectExpr("percentile(SALARY_FROM, 0.5) as m").collect()[0]["m"]
median_to = df.filter((col("SALARY_TO").isNotNull()) & (col("SALARY_TO") > 0)) \
               .selectExpr("percentile(SALARY_TO, 0.5) as m").collect()[0]["m"]
median_salary = df.filter((col("SALARY").isNotNull()) & (col("SALARY") > 0)) \
                   .selectExpr("percentile(SALARY, 0.5) as m").collect()[0]["m"]

print("Medians:", median_from, median_to, median_salary)

Medians: 88000.0 131040.0 116300.0


### 1.3 Imputing missing salaries

In [4]:
from pyspark.sql.functions import when

imputed_value = (median_from + median_to) / 2  # 108668.5

df = df.withColumn(
    "Average_Salary",
    when(col("SALARY").isNull(), imputed_value).otherwise(col("SALARY"))
)

# Also fill SALARY itself with median_salary where missing (matches the expected table)
df = df.withColumn(
    "SALARY",
    when(col("SALARY").isNull(), median_salary).otherwise(col("SALARY"))
)

In [5]:
df.select("Average_Salary", "SALARY", "EDUCATION_LEVELS_NAME", "REMOTE_TYPE_NAME",
          "MAX_YEARS_EXPERIENCE", "LOT_V6_SPECIALIZED_OCCUPATION_NAME").show(5, truncate=False)

+--------------+--------+-----------------------------+----------------+--------------------+----------------------------------+
|Average_Salary|SALARY  |EDUCATION_LEVELS_NAME        |REMOTE_TYPE_NAME|MAX_YEARS_EXPERIENCE|LOT_V6_SPECIALIZED_OCCUPATION_NAME|
+--------------+--------+-----------------------------+----------------+--------------------+----------------------------------+
|109520.0      |116300.0|[\n  "Bachelor's degree"\n]  |[None]          |2.0                 |General ERP Analyst / Consultant  |
|109520.0      |116300.0|[\n  "No Education Listed"\n]|Remote          |3.0                 |Oracle Consultant / Analyst       |
|109520.0      |116300.0|[\n  "Bachelor's degree"\n]  |[None]          |NULL                |Data Analyst                      |
|109520.0      |116300.0|[\n  "No Education Listed"\n]|[None]          |NULL                |Data Analyst                      |
|92500.0       |92500.0 |[\n  "No Education Listed"\n]|[None]          |NULL                |Orac

### 1.4 Cleaning Education column

In [6]:
from pyspark.sql.functions import regexp_replace

df = df.withColumn(
    "EDUCATION_LEVELS_NAME",
    regexp_replace(col("EDUCATION_LEVELS_NAME"), r"[\n\r]", "")
)

### 1.5 Exporting cleaned data

In [7]:
df_clean = df.dropna(subset=["Average_Salary"])

df_clean.coalesce(1).write.mode("overwrite").option("header", "true").csv("../data/lightcast_cleaned")

print(f"Data cleaning complete. Rows retained: {df_clean.count()}")

Data cleaning complete. Rows retained: 72498


## 2 Salary Distribution by Industry and Employment Type

In [8]:
import plotly.io as pio
import plotly.graph_objects as go

# Custom theme — personalize colors/fonts as you like
pio.templates["custom"] = go.layout.Template(
    layout={
        "title": {
            "font": {
                "family": "Georgia, serif",
                "size": 22,
                "color": "#1f2d3d"
            },
            "x": 0.5
        },
        "font": {
            "family": "Helvetica Neue, Helvetica, Arial, sans-serif",
            "size": 13,
            "color": "#1f2d3d"
        },
        "colorway": ["#2E86AB", "#E63946", "#F4A261", "#2A9D8F", "#8E44AD", "#F6C85F"],
        "plot_bgcolor": "#F7F9FB",
        "paper_bgcolor": "#FFFFFF",
        "hovermode": "closest"
    }
)

pio.templates.default = "custom"

In [13]:
import plotly.express as px

# Filter: remove rows where SALARY_FROM is missing or zero
section2_df = df.filter(
    (col("SALARY_FROM").isNotNull()) &
    (col("SALARY_FROM") > 0) &
    (col("NAICS2_NAME").isNotNull()) &
    (col("EMPLOYMENT_TYPE_NAME").isNotNull())
).select("NAICS2_NAME", "SALARY_FROM", "EMPLOYMENT_TYPE_NAME")

# Convert to pandas for Plotly (this subset is small enough)
section2_pdf = section2_df.toPandas()

# Build the box plot
fig = px.box(
    section2_pdf,
    x="NAICS2_NAME",
    y="SALARY_FROM",
    color="EMPLOYMENT_TYPE_NAME",
    title="Salary Distribution by Industry (NAICS2) and Employment Type",
    labels={
        "NAICS2_NAME": "Industry (NAICS2)",
        "SALARY_FROM": "Salary (Lower Bound, USD)",
        "EMPLOYMENT_TYPE_NAME": "Employment Type"
    },
    color_discrete_map={
        "Full-time (> 32 hours)": "#1B4965",   # deep navy
        "Part-time / full-time": "#5FA8D3",    # sky blue
        "Part-time (≤ 32 hours)": "#CAE9FF"    # pale blue
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=650,
    margin=dict(b=200),  # extra bottom margin for long industry names
    legend=dict(orientation="h", yanchor="bottom", y=-0.6, xanchor="center", x=0.5)
)
fig.update_yaxes(range=[0, 300000])   

fig.show()

I noticed that most median starting salaries fall somewhere between $60K and $100K across NAICS2 industries. Sectors like Construction, Finance and Insurance, Information, and Utilities tend to be on the higher end and also have a wider range of higher salaries. On the other hand, industries like Accommodation/Food Services, Retail Trade, and Arts/Entertainment are clearly lower.
I also saw that full-time roles (over 32 hours) usually pay more than part-time ones within each industry, but that difference isn’t as big in lower-paying sectors. So overall, it feels like the industry itself has a bigger impact on salary than just whether the job is full-time or part-time.

## 3 Salary Analysis by ONET Occupation Type (Bubble Chart)

In [17]:
print("ONET_NAME distinct:", df.select("ONET_NAME").distinct().count())
print("LOT_V6_SPECIALIZED_OCCUPATION_NAME distinct:", df.select("LOT_V6_SPECIALIZED_OCCUPATION_NAME").distinct().count())
print("LOT_OCCUPATION_NAME distinct:", df.select("LOT_OCCUPATION_NAME").distinct().count())
print("TITLE_NAME distinct:", df.select("TITLE_NAME").distinct().count())
print("SOC_4_NAME distinct:", df.select("SOC_4_NAME").distinct().count())

ONET_NAME distinct: 2


LOT_V6_SPECIALIZED_OCCUPATION_NAME distinct: 12


LOT_OCCUPATION_NAME distinct: 7


TITLE_NAME distinct: 5720


SOC_4_NAME distinct: 2


In [18]:
from pyspark.sql.functions import count as spark_count

section3_df = df.filter(
    (col("LOT_V6_SPECIALIZED_OCCUPATION_NAME").isNotNull()) &
    (col("SALARY_FROM").isNotNull()) &
    (col("SALARY_FROM") > 0)
).groupBy("LOT_V6_SPECIALIZED_OCCUPATION_NAME").agg(
    expr("percentile(SALARY_FROM, 0.5) as Median_Salary"),
    spark_count("*").alias("Job_Postings")
).orderBy(col("Job_Postings").desc())

section3_pdf = section3_df.toPandas()
print("Total occupations:", len(section3_pdf))
section3_pdf

Total occupations: 11


,LOT_V6_SPECIALIZED_OCCUPATION_NAME,Median_Salary,Job_Postings
0,Data Analyst,77040.0,13311
1,General ERP Analyst / Consultant,94000.0,3746
2,Oracle Consultant / Analyst,104000.0,3584
3,SAP Analyst / Admin,97875.0,3497
4,Enterprise Architect,130000.0,3376
5,Business Analyst (General),72250.0,1857
6,Business Intelligence Analyst,85143.0,1840
7,Financial Data Analyst,43680.0,521
8,Data Quality Analyst,80496.0,503
9,Healthcare Analyst,70000.0,96


In [19]:
fig = px.scatter(
    section3_pdf,
    x="LOT_V6_SPECIALIZED_OCCUPATION_NAME",
    y="Median_Salary",
    size="Job_Postings",
    color="Median_Salary",
    hover_name="LOT_V6_SPECIALIZED_OCCUPATION_NAME",
    hover_data={"Job_Postings": True, "Median_Salary": ":$,.0f"},
    title="Median Salary by Specialized Occupation (Bubble Size = Job Postings)",
    labels={
        "LOT_V6_SPECIALIZED_OCCUPATION_NAME": "Specialized Occupation",
        "Median_Salary": "Median Salary (USD)",
        "Job_Postings": "Number of Postings"
    },
    size_max=70,
    color_continuous_scale=[
        [0.0, "#2E86AB"],
        [0.5, "#F4A261"],
        [1.0, "#E63946"]
    ]
)

fig.update_layout(
    xaxis_tickangle=-35,
    height=650,
    margin=dict(b=220),
    showlegend=False
)

fig.show()

One thing that stands out is that Enterprise Architect has the highest median salary (around $130K), even though there aren’t that many job postings for it. Meanwhile, Data Analyst roles are by far the most common (over 13,000 postings), but the pay is much lower, closer to $77K.
What’s interesting here is that the more common a role is, the lower the salary tends to be. That makes me think that the most in-demand positions are mostly entry, or mid-level generalist roles, while more specialized jobs like Oracle, SAP, or Enterprise Architect are less common but pay significantly more.

## 4 Salary by Education Level

In [20]:
from pyspark.sql.functions import when

# Map raw education values into 4 clean groups
df_edu = df.withColumn(
    "EDU_GROUP",
    when(col("EDUCATION_LEVELS_NAME").contains("PhD"), "PhD")
    .when(col("EDUCATION_LEVELS_NAME").contains("Doctorate"), "PhD")
    .when(col("EDUCATION_LEVELS_NAME").contains("professional degree"), "PhD")
    .when(col("EDUCATION_LEVELS_NAME").contains("Master"), "Master")
    .when(col("EDUCATION_LEVELS_NAME").contains("Bachelor"), "Bachelor")
    .otherwise("Associate or Lower")
)

# Filter to rows with salary and experience info
section4_df = df_edu.filter(
    (col("Average_Salary").isNotNull()) &
    (col("MAX_YEARS_EXPERIENCE").isNotNull()) &
    (col("LOT_V6_SPECIALIZED_OCCUPATION_NAME").isNotNull())
).select(
    "MAX_YEARS_EXPERIENCE",
    "Average_Salary",
    "EDU_GROUP",
    "LOT_V6_SPECIALIZED_OCCUPATION_NAME"
)

section4_pdf = section4_df.toPandas()

# Check group counts
print(section4_pdf["EDU_GROUP"].value_counts())
section4_pdf.head()

EDU_GROUP
Bachelor              4576
Associate or Lower    1922
Master                1676
PhD                    256
Name: count, dtype: int64


,MAX_YEARS_EXPERIENCE,Average_Salary,EDU_GROUP,LOT_V6_SPECIALIZED_OCCUPATION_NAME
0,2.0,109520.0,Bachelor,General ERP Analyst / Consultant
1,3.0,109520.0,Associate or Lower,Oracle Consultant / Analyst
2,7.0,109520.0,Associate or Lower,General ERP Analyst / Consultant
3,2.0,92962.0,Master,Data Analyst
4,5.0,109520.0,Master,Data Analyst


In [21]:
import numpy as np

# Add jitter to experience so points don't stack on integer values
np.random.seed(42)
section4_pdf["MAX_YEARS_EXPERIENCE_JITTER"] = (
    section4_pdf["MAX_YEARS_EXPERIENCE"] + np.random.uniform(-0.3, 0.3, size=len(section4_pdf))
)

edu_order = ["Associate or Lower", "Bachelor", "Master", "PhD"]
edu_colors = {
    "Associate or Lower": "#2E86AB",
    "Bachelor": "#F4A261",
    "Master": "#E63946",
    "PhD": "#6A4C93"
}

for group in edu_order:
    subset = section4_pdf[section4_pdf["EDU_GROUP"] == group]
    if len(subset) == 0:
        print(f"No data for {group}, skipping.")
        continue

    fig = px.scatter(
        subset,
        x="MAX_YEARS_EXPERIENCE_JITTER",
        y="Average_Salary",
        color_discrete_sequence=[edu_colors[group]],
        hover_name="LOT_V6_SPECIALIZED_OCCUPATION_NAME",
        hover_data={"MAX_YEARS_EXPERIENCE_JITTER": False, "Average_Salary": ":$,.0f"},
        title=f"Salary vs Experience — {group} (n={len(subset)})",
        labels={
            "MAX_YEARS_EXPERIENCE_JITTER": "Max Years Experience (jittered)",
            "Average_Salary": "Average Salary (USD)"
        },
        opacity=0.5
    )

    fig.update_traces(marker=dict(size=7))
    fig.update_layout(height=500, showlegend=False)
    fig.show()

Looking at the data, salaries do go up a bit as experience increases across all education levels, but not as much as I would’ve expected. There’s actually a lot of variation within each group, so experience by itself doesn’t seem to be a strong predictor of salary.
Bachelor’s degrees show up the most (over 4,500 postings) and also have the widest range of salaries. PhD roles, on the other hand, are pretty rare (around 250), but they tend to be grouped at higher salary levels, especially for more senior positions.
One thing to keep in mind is that there’s a noticeable horizontal line around $109K across all groups. That comes from imputed salaries in Section 1.3, so the real differences are better understood by looking at the points above and below that line.

## 5 Salary by Remote Work Type

In [22]:
# Map REMOTE_TYPE_NAME into 3 clean groups
df_remote = df.withColumn(
    "REMOTE_GROUP",
    when(col("REMOTE_TYPE_NAME") == "Remote", "Remote")
    .when(col("REMOTE_TYPE_NAME") == "Hybrid Remote", "Hybrid")
    .when(col("REMOTE_TYPE_NAME").contains("Hybrid"), "Hybrid")
    .otherwise("Onsite")  # catches [None], blanks, "Not Remote", etc.
)

section5_df = df_remote.filter(
    (col("Average_Salary").isNotNull()) &
    (col("MAX_YEARS_EXPERIENCE").isNotNull()) &
    (col("LOT_V6_SPECIALIZED_OCCUPATION_NAME").isNotNull())
).select(
    "MAX_YEARS_EXPERIENCE",
    "Average_Salary",
    "LOT_V6_SPECIALIZED_OCCUPATION_NAME",
    "REMOTE_GROUP"
)

section5_pdf = section5_df.toPandas()

# Verify groups
print(section5_pdf["REMOTE_GROUP"].value_counts())

REMOTE_GROUP
Onsite    6535
Remote    1612
Hybrid     283
Name: count, dtype: int64


In [23]:
df.groupBy("REMOTE_TYPE_NAME").count().show(truncate=False)

+----------------+-----+
|REMOTE_TYPE_NAME|count|
+----------------+-----+
|Remote          |12497|
|[None]          |56570|
|NULL            |44   |
|Not Remote      |1127 |
|Hybrid Remote   |2260 |
+----------------+-----+



In [24]:
# Add jitter for experience
np.random.seed(42)
section5_pdf["MAX_YEARS_EXPERIENCE_JITTER"] = (
    section5_pdf["MAX_YEARS_EXPERIENCE"] + np.random.uniform(-0.3, 0.3, size=len(section5_pdf))
)

remote_order = ["Remote", "Hybrid", "Onsite"]
remote_colors = {
    "Remote": "#2A9D8F",   # teal
    "Hybrid": "#F4A261",   # warm orange
    "Onsite": "#E63946"    # red
}

for group in remote_order:
    subset = section5_pdf[section5_pdf["REMOTE_GROUP"] == group]
    if len(subset) == 0:
        print(f"No data for {group}, skipping.")
        continue

    fig = px.scatter(
        subset,
        x="MAX_YEARS_EXPERIENCE_JITTER",
        y="Average_Salary",
        color_discrete_sequence=[remote_colors[group]],
        hover_name="LOT_V6_SPECIALIZED_OCCUPATION_NAME",
        hover_data={"MAX_YEARS_EXPERIENCE_JITTER": False, "Average_Salary": ":$,.0f"},
        title=f"Salary vs Experience — {group} (n={len(subset)})",
        labels={
            "MAX_YEARS_EXPERIENCE_JITTER": "Max Years Experience (jittered)",
            "Average_Salary": "Average Salary (USD)"
        },
        opacity=0.5
    )
    fig.update_traces(marker=dict(size=7))
    fig.update_layout(height=500, showlegend=False)
    fig.show()

In [25]:
for group in remote_order:
    subset = section5_pdf[section5_pdf["REMOTE_GROUP"] == group]
    if len(subset) == 0:
        continue

    fig = px.histogram(
        subset,
        x="Average_Salary",
        nbins=40,
        color_discrete_sequence=[remote_colors[group]],
        title=f"Salary Distribution — {group} (n={len(subset)})",
        labels={"Average_Salary": "Average Salary (USD)", "count": "Number of Postings"}
    )
    fig.update_layout(height=450, bargap=0.05, showlegend=False)
    fig.show()

What surprised me most is how dominant onsite roles are compared to remote and hybrid combined (6,535 vs 1,895). Across all three, salaries tend to cluster around that ~$109K mark, with a few higher-paying roles stretching past $250K.
There’s a slight trend where remote roles reach higher salaries more often than onsite ones, which might suggest a small remote premium. Hybrid roles are pretty rare (only 283 postings), so the data there is quite scattered and harder to interpret.
Just like with the education breakdown, experience doesn’t seem to have a huge impact on salary here either. It feels like the type of role itself plays a much bigger part in determining pay.